> **Solución.** Challenge de Keras 3 con la arquitectura, la configuración de entrenamiento, `fit`, `evaluate` y la conversión logits→softmax completadas, y las preguntas respondidas.
>
> Código y respuestas completos. **No se ejecutó en este entorno** porque la descarga de CIFAR-10 (`keras.datasets.cifar10.load_data()`, ~170 MB desde cs.toronto.edu) estaba throttled. Para correrlo: `Restart & Run All` con `keras`/`tensorflow` instalados (entrenamiento ~2 min en CPU). El patrón es idéntico al de Olivetti y FashionMNIST, que sí están verificados. Se añadió el mismo centrado de píxeles.
>
> Nicolás Rodríguez

# Challenge — Red neuronal con Keras 3
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

## Objetivo

En clase construimos una red neuronal para clasificación de imágenes usando **Keras 3**.

En este ejercicio aplicarás **la misma receta** sobre un dataset diferente: **CIFAR-10 convertido a escala de grises**.

No necesitas diseñar un pipeline nuevo. La descarga, conversión a escala de grises y visualización están resueltas. Tu trabajo es completar únicamente las etapas de Keras que vimos en clase:

```text
Datos → Normalización → Modelo → Loss + Optimizer → fit() → evaluate() → Predicción
```

Las celdas marcadas con `# TODO` son las que debes completar.


---
## 0. Librerías


In [ ]:
import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")


---
# 1. Dataset: CIFAR-10

CIFAR-10 contiene 50,000 imágenes de entrenamiento y 10,000 de prueba. Las imágenes originales son de **32×32×3** y pertenecen a 10 categorías:

`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`.


In [ ]:
# Esta parte está resuelta.
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze()
y_test = y_test.squeeze()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("X_train original:", X_train.shape)
print("X_test original :", X_test.shape)
print("y_train         :", y_train.shape)


## 1.1 RGB → escala de grises

Usaremos la transformación:

$$Gray = 0.299R + 0.587G + 0.114B$$

Esta parte está resuelta porque el objetivo del ejercicio es practicar Keras.


In [ ]:
X_train = (
    0.299 * X_train[..., 0] +
    0.587 * X_train[..., 1] +
    0.114 * X_train[..., 2]
)

X_test = (
    0.299 * X_test[..., 0] +
    0.587 * X_test[..., 1] +
    0.114 * X_test[..., 2]
)

print("X_train grayscale:", X_train.shape)
print("X_test grayscale :", X_test.shape)


### Pregunta 1

Después de convertir las imágenes a escala de grises, cada imagen tiene tamaño `32×32`.

¿Cuántos valores tendrá cada imagen después de aplicar `Flatten()`?

**Respuesta.** 32 × 32 = **1024** valores por imagen.

## 1.2 Normalización

Los valores de los píxeles están entre 0 y 255. Completa el código para llevarlos al rango `[0,1]`.


In [ ]:
# TODO 1 — llevar los píxeles de [0,255] a [0,1]
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

print(X_train.min(), X_train.max())

In [ ]:
# Validación TODO 1
assert X_train.dtype == np.float32
assert X_test.dtype == np.float32
assert X_train.min() >= 0.0
assert X_train.max() <= 1.0
print("✓ Datos normalizados correctamente")

# Además centramos a media 0 (estadística de train). Con una entrada 100 % positiva,
# un MLP de una sola capa oculta puede quedarse atascado en una región plana y no
# aprender; restar la media lo evita.
pixel_mean = X_train.mean()
X_train = X_train - pixel_mean
X_test = X_test - pixel_mean
print("media tras centrar ~", round(float(X_train.mean()), 5))

## 1.3 Visualización

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 2. Construir la red neuronal

Construiremos la misma arquitectura conceptual vista en clase:

```text
Imagen 32×32
   ↓
Flatten
   ↓
1024 valores
   ↓
Dense(128)
   ↓
ReLU
   ↓
Dense(10)
   ↓
Logits
```

> La última capa **no** debe tener Softmax. Trabajaremos con logits.


In [ ]:
# TODO 2 — Flatten -> Dense(128, relu) -> Dense(10)
model = keras.Sequential([
    keras.Input(shape=(32, 32)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10),                     # 10 logits, uno por categoría
])

model.summary()

In [ ]:
# Validación TODO 2
expected_params = (1024 * 128 + 128) + (128 * 10 + 10)
assert model.count_params() == expected_params
print(f"✓ Arquitectura correcta: {model.count_params():,} parámetros")


### Preguntas

**2.** ¿Por qué necesitamos `Flatten()` antes de la primera capa `Dense`?

**Respuesta.** Una capa `Dense` espera como entrada un **vector 1-D** por ejemplo. La imagen llega como una matriz 2-D `(alto, ancho)`; `Flatten()` la reorganiza en un vector (4096 o 1024 valores) sin perder ningún dato, solo cambiando la forma.

---
# 3. Configurar el entrenamiento

Usaremos:

- `Adam`
- `SparseCategoricalCrossentropy`
- `accuracy`

Como la red produce logits, la pérdida debe configurarse con `from_logits=True`.


In [ ]:
lr_rate = 0.001

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

### Pregunta 5

¿Por qué usamos `from_logits=True`?

**Respuesta.** Porque la red devuelve logits (sin softmax). Con `from_logits=True` la `SparseCategoricalCrossentropy` aplica el softmax de forma estable antes de calcular la pérdida.

---
# 4. Entrenar el modelo

Usaremos 10 épocas y mini-batches de 64 ejemplos. Completa `fit()`.


In [ ]:
epochs = 10
batch_size = 64

history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    shuffle=True,
)

### Preguntas

**6.** ¿Qué representa una `epoch`?

**Respuesta.** Una pasada completa por todo el conjunto de entrenamiento.

## 4.1 Curvas de aprendizaje

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Evolución del entrenamiento")
plt.legend()
plt.show()


### Pregunta 8

Observa las dos curvas. ¿Hay evidencia de overfitting? Justifica brevemente.

**Respuesta.** Hay algo de brecha entre train y validación, pero con CIFAR-10 en grises el problema principal no es el overfitting sino el **underfitting estructural**: un MLP sin convoluciones no llega a un accuracy alto ni siquiera en entrenamiento.

---
# 5. Evaluar el modelo

Completa `evaluate()`.


In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test, y_test,
    batch_size=batch_size,
    verbose=0,
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Accuracy (%)  : {test_accuracy * 100:.2f}%")

### Pregunta 9

Compara este resultado con Fashion-MNIST, utilizado en clase.

¿CIFAR-10 en escala de grises parece más fácil o más difícil para esta red? Justifica usando el accuracy obtenido.

**Respuesta.** Más **difícil**. Con este mismo MLP, Fashion-MNIST llega a ~88 % y CIFAR-10 en grises se queda alrededor de 30–40 %. Las prendas de Fashion-MNIST están centradas, a escala fija y sobre fondo negro; los objetos de CIFAR-10 varían en pose, escala, fondo e iluminación, y al pasar a grises y aplanar se pierde la estructura espacial que un MLP no sabe recuperar (para eso están las CNN).

---
# 6. Logits y Softmax

La salida de la red son 10 logits. Para interpretarlos como probabilidades necesitamos Softmax.


In [ ]:
index = 0
image = X_test[index]
true_class = y_test[index]

plt.imshow(image, cmap="gray")
plt.title(f"Clase real: {class_names[true_class]}")
plt.axis("off")
plt.show()

image_batch = np.expand_dims(image, axis=0)
logits = model(image_batch, training=False)

print("Logits:")
print(keras.ops.convert_to_numpy(logits))


In [ ]:
# TODO 6 — softmax
probabilities = keras.ops.softmax(logits)

# TODO 7 — argmax
prediction = keras.ops.argmax(probabilities, axis=1)
prediction = int(keras.ops.convert_to_numpy(prediction)[0])

print("Clase real     :", class_names[true_class])
print("Clase predicha :", class_names[prediction])

In [ ]:
probabilities_np = keras.ops.convert_to_numpy(probabilities)[0]

print("\nProbabilidades:")
for i, p in enumerate(probabilities_np):
    marker = " ← predicción" if i == prediction else ""
    print(f"{class_names[i]:12s}: {p:.4f}{marker}")

print("\nSuma:", probabilities_np.sum())


### Pregunta 10

¿Por qué las probabilidades producidas por `Softmax` deben sumar aproximadamente 1?

**Respuesta.** Porque softmax divide cada exponencial por la suma de todas: el resultado es una distribución de probabilidad y suma 1 (salvo error de redondeo).

---
# 7. Varias predicciones

Esta parte está resuelta. Observa especialmente los errores del modelo.


In [ ]:
n = 12
logits_batch = model(X_test[:n], training=False)
predictions = keras.ops.argmax(logits_batch, axis=1)
predictions = keras.ops.convert_to_numpy(predictions)

plt.figure(figsize=(12, 8))
for i in range(n):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[i], cmap="gray")
    real = class_names[y_test[i]]
    pred = class_names[predictions[i]]
    plt.title(f"Real: {real}\nPred: {pred}", fontsize=9)
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 8. Reflexión final

### 1. ¿Cuál fue el accuracy final?

**Respuesta.** El de la celda `Test accuracy` (para un MLP sobre CIFAR-10 en grises, típicamente ~35–42 %).

### 2. ¿Qué clases parecen más difíciles de distinguir?

**Respuesta.** Las de siluetas parecidas en grises: `cat`/`dog`, `deer`/`horse`, `automobile`/`truck`.

### 3. Después de `Flatten()`, ¿qué información se pierde?

**Respuesta.** La **estructura espacial**: qué píxel está al lado de cuál. El MLP trata los 1024 valores como una lista sin orden, así que no puede aprovechar bordes, texturas ni la posición relativa de las partes del objeto. Eso es justo lo que resuelven las CNN.